# Monte Carlo Simulation Demo

Random walks, European option pricing, and Asian option pricing.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.pardir, "src"))

import numpy as np
import matplotlib.pyplot as plt

from nms.monte_carlo import random_walk_1d, random_walk_2d, european_option_mc, asian_option_mc

## 1. Random Walks

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1-D random walks
paths_1d = random_walk_1d(500, n_walks=10, seed=42)
for path in paths_1d:
    axes[0].plot(path, alpha=0.7, lw=0.8)
axes[0].set_title("1-D Random Walks (10 paths, 500 steps)")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Position")
axes[0].grid(True, alpha=0.3)

# 2-D random walks
paths_2d = random_walk_2d(1000, n_walks=5, seed=42)
for i in range(5):
    axes[1].plot(paths_2d[i, :, 0], paths_2d[i, :, 1], alpha=0.7, lw=0.5)
    axes[1].plot(paths_2d[i, 0, 0], paths_2d[i, 0, 1], "ko", ms=4)
    axes[1].plot(paths_2d[i, -1, 0], paths_2d[i, -1, 1], "r^", ms=5)
axes[1].set_title("2-D Random Walks (5 paths, 1000 steps)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
axes[1].set_aspect("equal")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. European Option Pricing

Price a European call via Monte Carlo and compare to Black-Scholes.

In [ ]:
from scipy.stats import norm

def black_scholes_call(S, K, r, sigma, T):
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

S0, K, r, sigma, T = 100, 105, 0.05, 0.2, 1.0

bs_price = black_scholes_call(S0, K, r, sigma, T)
mc_result = european_option_mc(S0, K, r, sigma, T, n_paths=500_000, seed=0)

print(f"Black-Scholes price:  {bs_price:.4f}")
print(f"Monte Carlo price:    {mc_result.price:.4f} +/- {mc_result.std_error:.4f}")
print(f"95% CI:               [{mc_result.ci_lower:.4f}, {mc_result.ci_upper:.4f}]")
print(f"Paths simulated:      {mc_result.n_paths:,}")

## 3. Convergence with Number of Paths

In [ ]:
path_counts = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000]
mc_prices = []
mc_errors = []

for n in path_counts:
    result = european_option_mc(S0, K, r, sigma, T, n_paths=n, seed=0)
    mc_prices.append(result.price)
    mc_errors.append(result.std_error)

plt.figure(figsize=(10, 5))
plt.errorbar(path_counts, mc_prices, yerr=[1.96 * e for e in mc_errors],
             fmt="o-", capsize=3, label="MC estimate")
plt.axhline(bs_price, color="red", ls="--", label=f"Black-Scholes = {bs_price:.4f}")
plt.xscale("log")
plt.xlabel("Number of paths")
plt.ylabel("Option price")
plt.title("Monte Carlo Convergence")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Asian Option

In [ ]:
asian = asian_option_mc(S0, K, r, sigma, T, n_steps=252, n_paths=200_000, seed=0)
print(f"Asian call price:  {asian.price:.4f} +/- {asian.std_error:.4f}")
print(f"European call:     {mc_result.price:.4f}")
print(f"Asian < European:  {asian.price < mc_result.price}")